In [1]:
import pandas as pd 
import sqlalchemy as sal
import pyodbc

In [4]:
engine = sal.create_engine('mssql://Sudhanshu\\SQLEXPRESS/namastesql?driver=ODBC+DRIVER+17+FOR+SQL+SERVER')
conn=engine.connect()


In [6]:
df_products_db = pd.read_sql_query('select category ,sum(sales) as Total_sales from orders group by category' ,conn)
df_products_db

,category,Total_sales
0,Office Supplies,719047.0450
1,Furniture,741999.7774
2,Technology,836154.1490


In [39]:
def extract():
    df_orders = pd.read_csv('orders.txt')
    df_returns = pd.read_csv('returns.txt')
    return df_orders , df_returns

def transform(df_orders,df_returns):
    df = pd.merge(df_orders,df_returns, how = 'inner' , left_on='order_id' , right_on= 'order_id')
    return df

def load(df):
    df.to_sql('orders_final',con=conn , index=False , if_exists = 'append')
    conn.commit()

In [40]:
#extract()

In [41]:
def transform(df_orders,df_returns):
    df = pd.merge(df_orders,df_returns, how='inner' , left_on='order_id',right_on='order_id')
    return df


In [42]:
#load 
def load(df):
    df.to_sql('orders_final',con=conn,index=False,if_exists='replace') #inplace of replace we also use append
    conn.commit()
    

In [26]:
#extract
df_orders , df_returns = extract()

#transform
df = transform(df_orders,df_returns)

#
load(df)


In [27]:
df

,order_id,order_date,customer_name,city,category,product_id,sales,profit,Return Reason
0,CA-2018-100762,2018-11-24,Nat Gilpin,Jackson,Office Supplies,OFF-AR-10000380,151.920,45.5760,Bad Quality
1,CA-2018-100867,2018-10-19,Eugene Hildebrand,Lakewood,Technology,TEC-PH-10004922,321.552,20.0970,Bad Quality
2,CA-2018-102652,2018-04-06,Andy Yotov,Los Angeles,Furniture,FUR-FU-10000747,91.960,15.6332,Bad Quality
3,CA-2018-103373,2018-05-18,Bruce Stewart,Cleveland,Technology,TEC-PH-10002885,779.796,-168.9558,Bad Quality
4,CA-2018-103744,2018-02-23,Michael Grace,El Paso,Office Supplies,OFF-BI-10000320,4.428,-6.8634,Bad Quality
...,...,...,...,...,...,...,...,...,...
291,US-2021-136679,2021-11-14,Xylona Preis,Pasadena,Office Supplies,OFF-AR-10003582,45.040,4.5040,Others
292,US-2021-147886,2021-03-28,Dave Hallsten,Fairfield,Furniture,FUR-FU-10001095,26.480,10.0624,Others
293,US-2021-147998,2021-05-19,Sue Ann Reed,San Jose,Office Supplies,OFF-BI-10002082,133.120,49.9200,Wrong Items
294,US-2021-151127,2021-05-22,Rob Lucas,Los Angeles,Office Supplies,OFF-AR-10002445,49.560,18.8328,Wrong Items


In [28]:
df_sql = pd.read_sql_query('select * from orders_final',conn)
df_sql

,order_id,order_date,customer_name,city,category,product_id,sales,profit,Return Reason
0,CA-2018-100762,2018-11-24,Nat Gilpin,Jackson,Office Supplies,OFF-AR-10000380,151.920,45.5760,Bad Quality
1,CA-2018-100867,2018-10-19,Eugene Hildebrand,Lakewood,Technology,TEC-PH-10004922,321.552,20.0970,Bad Quality
2,CA-2018-102652,2018-04-06,Andy Yotov,Los Angeles,Furniture,FUR-FU-10000747,91.960,15.6332,Bad Quality
3,CA-2018-103373,2018-05-18,Bruce Stewart,Cleveland,Technology,TEC-PH-10002885,779.796,-168.9558,Bad Quality
4,CA-2018-103744,2018-02-23,Michael Grace,El Paso,Office Supplies,OFF-BI-10000320,4.428,-6.8634,Bad Quality
...,...,...,...,...,...,...,...,...,...
291,US-2021-136679,2021-11-14,Xylona Preis,Pasadena,Office Supplies,OFF-AR-10003582,45.040,4.5040,Others
292,US-2021-147886,2021-03-28,Dave Hallsten,Fairfield,Furniture,FUR-FU-10001095,26.480,10.0624,Others
293,US-2021-147998,2021-05-19,Sue Ann Reed,San Jose,Office Supplies,OFF-BI-10002082,133.120,49.9200,Wrong Items
294,US-2021-151127,2021-05-22,Rob Lucas,Los Angeles,Office Supplies,OFF-AR-10002445,49.560,18.8328,Wrong Items


In [33]:
def extract():
    df_products = pd.read_csv('products.txt')
    df_products_db = pd.read_sql_query('select * from products' ,conn)
    return df_products,df_products_db

def transform(df_products, df_products_db):
    df_merge = pd.merge(
        df_products,
        df_products_db,
        how='left',
        on='product_id'
    )
    
    df_insert = df_merge[df_merge['product_name_y'].isna()]
    df_insert_final = df_insert.iloc[0 , 0:3]
    return df_insert_final

def load(df_insert_final):
    df.to_sql('products',con=conn,index=False,if_exists='append') #inplace of replace we also use append
    conn.commit()

    

In [34]:
df_products,df_products_db = extract()

In [35]:
df_insert_final = transform(df_products,df_products_db)

In [36]:
load(df_insert_final)

ProgrammingError: (pyodbc.ProgrammingError) ('42S22', "[42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'order_id'. (207) (SQLExecDirectW); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'order_date'. (207); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'customer_name'. (207); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'city'. (207); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'category'. (207); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'sales'. (207); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'profit'. (207); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'Return Reason'. (207); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid column name 'order_id'. (207); [42S22] [Microso